# Comparing p53 DNA-damage response modelling and expression-based signatures for breast cancer doxorubicin response and survival

## Approach taken

This assignment focuses on breast cancer, using data from The Cancer Genome Atlas (TCGA), the DepMap cell-line database, and the PRISM drug response database.

Four of the six questions (Q1 to Q4) have been attempted. These were chosen because together they allow a direct comparison of a mechanistic p53 ODE approach with expression based regression approaches across both endpoints, patient survival and cell-line drug response. This reflects my curiosity about the techniques involved and my interest in understanding what an end to end review of breast cancer and doxorubicin treatment would surface from publicly available data.  

Two AI-based tools (Cursor 3.8.11 and GPT-5.5) were used during this assignment, which also helped in answering the selected questions within the timeframe given. Specifically, AI assistance was used to support exploratory data analysis, to help polish the structure and wording of the final report notebook, to debug the notebook to PDF rendering workflow, and to assist with routine GitHub repository and version control tasks.

All analytical decisions, modelling choices, interpretation of results, and final conclusions are my own. I have reviewed and verified all AI assisted content, and I take full responsibility for the accuracy and integrity of the work submitted.

## Abstract

This assignment compares mechanistic p53 DNA damage response modelling with expression based regression signatures in breast cancer. TCGA-BRCA tumour expression and clinical follow up were used for patient survival analyses, while DepMap/CCLE expression and PRISM doxorubicin viability response were used for breast cancer cell-line drug response analyses. A simplified p53 S46 ODE score from the given course template showed a modest continuous association with TCGA-BRCA overall survival (Cox HR 0.85 per standard deviation, 95% CI 0.73-0.99, p=0.038), which attenuated to borderline after adjusting for age and stage (HR 0.85, p=0.058). The same score showed no detectable association with PRISM doxorubicin sensitivity in 26 breast cancer cell lines (Spearman r=0.04, 95% CI -0.37 to 0.44). Expression based transfer models were also weak: a PRISM derived doxorubicin signature had moderate apparent fit but poor cross validation (in-sample R2=0.44, leave-one-out R2=-0.83) and did not predict TCGA survival (HR 0.95, 95% CI 0.78-1.15, p=0.60), and a TCGA-derived survival score, although strongly associated with survival within TCGA (HR 2.72, p<1e-7), did not predict PRISM doxorubicin response (Spearman r=0.07, 95% CI -0.35 to 0.46). Overall, the strongest results were within domain, and neither mechanistic nor expression based signatures transferred cleanly between patient survival and in vitro drug sensitivity, which are related but non equivalent endpoints.

## Submission note and code availability

This notebook was used to generate the final PDF report. Supporting notebooks contain the detailed data preparation and modelling steps and are submitted separately.

All code for this analysis is available at [https://github.com/Lifework-Health/TCGA-BRCA_pers_med](https://github.com/Lifework-Health/TCGA-BRCA_pers_med).

## Introduction

Breast cancer is a common and clinically heterogeneous cancer. Doxorubicin is an anthracycline chemotherapy used in breast cancer treatment, and its activity is closely linked to DNA damage, replication stress and cell death pathways. This makes doxorubicin a useful example for a personalised medicine assignment as the biological target process is plausible, but measured patient benefit depends on many factors beyond tumour cell intrinsic drug sensitivity.

The p53 pathway is central to the DNA damage response. After DNA damage, p53 can promote cell cycle arrest, DNA repair, senescence or apoptosis depending on cellular context and signal strength. Phosphorylation markers such as p53 S15 and p53 S46 are often interpreted as different aspects of p53 activation. This project therefore asks whether p53/DNA damage response features, simulated p53 ODE scores and expression based signatures can help explain either patient survival or doxorubicin response.

The main personalised medicine issue is endpoint transfer. TCGA-BRCA provides patient tumour expression and survival [1], but not direct doxorubicin response. PRISM provides in vitro doxorubicin viability response [3], but not patient outcomes. Comparing these sources tests whether a biologically motivated signal transfers between cell-line drug sensitivity and patient prognosis.

This report focuses on four questions (Q1 to Q4). They were chosen specifically so that the mechanistic p53 ODE model and expression-based regression signatures can be compared directly on the same survival and drug-response endpoints: Q1 and Q2 apply the mechanistic ODE score, while Q3 and Q4 apply expression-based regression signatures, so the two modelling philosophies are tested on a like-for-like basis rather than in a broader but shallower survey.

## Assignment Questions and Study Design

The analysis focuses on four questions, chosen so that a mechanistic p53 ODE model can be compared directly with expression-based regression signatures on the same survival and drug-response endpoints:

1. **Q1:** Are p53 ODE response scores associated with TCGA-BRCA patient survival?
2. **Q2:** Are p53 ODE response scores associated with PRISM doxorubicin sensitivity in breast cancer cell lines?
3. **Q3:** Can a PRISM derived doxorubicin expression signature be transferred to TCGA-BRCA survival?
4. **Q4:** Can a TCGA derived survival expression score be transferred to PRISM doxorubicin sensitivity?

Q1 and Q2 test the mechanistic ODE score against patient survival and cell-line drug response, while Q3 and Q4 test expression-based regression signatures across the same two endpoints, so the two modelling philosophies can be compared on a like-for-like basis. The original brief also included two further questions. The mechanistic-versus-ML comparison (Q5) is addressed directly by the Q1 to Q4 results, which contrast the mechanistic ODE score with expression-based regression signatures on the same survival and drug-response endpoints; a compact summary of that comparison is given before the Discussion. The LINCS feasibility question (Q6) was not attempted because no usable LINCS doxorubicin perturbation data were available in the working repository, and no fabricated result was produced.

The data sources answer different parts of the assignment. TCGA-BRCA is used for patient tumour expression and survival [1]. DepMap/CCLE is used for baseline breast cancer cell-line expression [2]. PRISM is used for doxorubicin viability response in cell lines [3]. The given p53 ODE/course template is used to simulate DNA damage response scores, especially p53 S15 and p53 S46 summaries across DDR inputs.

## Data and Preprocessing

The validated TCGA-BRCA table contained **1,094 patient level rows**, **151 deaths/events** and **943 censored patients** [1]. Age and stage were available as clinical covariates, and all 16 shared p53/DNA damage genes were present: ATM, CHEK2, HIPK2, MDM2, PPM1D, SIAH1, TP53, WSB1, CDKN1A, BAX, BBC3, GADD45A, MDM4, ATR, CHEK1 and CASP3. The Q1, Q3 and Q4 survival model tables used **1,081 complete model rows** after joining the relevant scores.

The validated PRISM/DepMap table contained **26 breast cancer cell lines** with baseline DepMap/CCLE expression and non missing PRISM doxorubicin response [2,3]. PRISM doxorubicin log-fold-change was interpreted as relative abundance or viability versus control, not as gene expression change. A more negative PRISM LFC means lower relative abundance after treatment, so the analysis defined a convenience sensitivity score where higher values mean greater apparent sensitivity.

The PRISM doxorubicin response was transformed into a sensitivity score as follows:

```text
relative_abundance_percent  = 100 * (2 ** prism_doxorubicin_lfc)
doxorubicin_sensitivity_score = -1 * prism_doxorubicin_lfc
```

The small PRISM/DepMap sample size came from intersecting several data requirements, not from the total number of breast cancer cell lines in DepMap.

| attrition_step | count | note |
| --- | --- | --- |
| Total rows/models in DepMap Model.csv | 1719 | ID column: ModelID |
| Models labelled as breast cancer / breast lineage | 71 | Filtered using OncotreeLineage and OncotreePrimaryDisease |
| Breast cancer models with expression data | 71 | Expression table contains 1719 total model IDs |
| Breast cancer models with selected p53/DNA-damage gene expression columns | 71 | Genes found: 16 of 16 |
| Doxorubicin treatment rows found in PRISM treatment info | 1 | Matched by searching name for doxorubicin |
| Breast cancer models with any PRISM response data | 26 | Any row in primary-screen-replicate-collapsed-logfold-change.csv |
| Breast cancer models with non-missing PRISM doxorubicin LFC | 26 | Doxorubicin LFC aggregated across 1 available treatment column(s) |
| Final joined modelling table rows | 26 | Intersection of breast metadata, selected expression genes, and non-missing PRISM doxorubicin response |

![PRISM/DepMap attrition funnel](../figures/prism_depmap_attrition_funnel.png)

*Figure 1. Formation of the PRISM/DepMap modelling cohort.*

## Statistical Methods

TCGA-BRCA survival analyses used Cox proportional hazards models [4]. Where appropriate, age adjusted Cox models were added to check whether the main score remained associated with survival after accounting for patient age. Median split Kaplan-Meier plots were used as visual summaries, but continuous Cox models were treated as the main survival analyses because dichotomising a continuous score loses information.

PRISM doxorubicin sensitivity was modelled using elastic net regression for the 16 shared p53/DNA damage genes [5]. Because the PRISM/DepMap breast cancer cohort contained only 26 cell lines, leave-one-out cross validation was used as a simple sensitivity check for generalisation. Spearman and Pearson correlations were used for PRISM transfer analyses to summarise monotonic and linear associations. For the transfer analyses (Q3 and Q4), expression was standardised within each domain to create relative within-domain signature scores, rather than reusing the training-domain scaler, because TCGA and DepMap expression are on different scales. This avoids directly comparing raw TCGA and DepMap expression, but it means the transferred scores should be interpreted as relative rankings within each domain rather than as calibrated predictions across domains. Transferred survival scores were additionally split with both a median and an optimal (minimum p-value) cutpoint, matching the course transfer scripts. Correlations are reported with approximate 95% confidence intervals (Fisher z-transformation), which are wide given the small cell-line cohort.

## p53 ODE Model Explanation

The p53 ODE analysis used the given course template as a pre specified mechanistic model of the DNA damage response. In the model, **DDR** means DNA damage response input, which is a simplified numerical signal representing increasing DNA damage stimulation. The model is simulated across a range of DDR doses and returns p53 response summaries.

The primary ODE feature was p53_S46_AUC_across_DDR, chosen a priori because p53 S46 is interpreted in this simplified model as a stronger stress/apoptosis-associated p53 response than p53 S15, which is treated as an earlier DNA damage activation marker. Since doxorubicin acts through DNA damage and downstream cell-death pathways, an integrated S46 response across DDR inputs was considered the most biologically relevant primary ODE summary. Other p53 ODE outputs were therefore treated as secondary or exploratory rather than as the main Q1 endpoint.



The conceptual p53 ODE scoring workflow, applied to each sample or cell line:

1. use baseline p53/DNA-damage gene expression to set the model inputs;
2. simulate the given p53 ODE template across DDR doses;
3. summarise the p53 S15 and p53 S46 response curves;
4. test the resulting score against survival or PRISM doxorubicin sensitivity.

## Results Q1: p53 ODE Model and TCGA-BRCA Survival

The pre-specified Q1 feature was `p53_S46_AUC_across_DDR`, and the primary analysis is the continuous Cox model [4]. Higher p53 S46 AUC was modestly associated with lower overall-survival hazard in TCGA-BRCA: **HR 0.85 per SD (95% CI 0.73-0.99), p=0.038**. The age-adjusted model was similar: **HR 0.84 (95% CI 0.72-0.98), p=0.027**. Adding tumour stage, which is itself strongly prognostic (**HR 2.23 per stage level, 95% CI 1.79-2.78, p<1e-12**), attenuated the p53 effect to borderline: **HR 0.85 (95% CI 0.72-1.01), p=0.058**. This is itself informative: it suggests the score partly tracks general disease severity rather than carrying fully stage-independent prognostic information. A Schoenfeld residual test gave no evidence of a proportional-hazards violation (**p=0.21**), so a single hazard ratio is a reasonable summary. A median-split Kaplan-Meier comparison was weaker (**log-rank p=0.102**), as expected when a continuous score is dichotomised at the median.

The optimal cutpoint and optimal DDR-dose analyses are retained only as exploratory visual sensitivity checks, because the threshold (and, for the dose scan, the dose) are selected from the same dataset to maximise separation. The optimal cutpoint suggested the association may be concentrated around a threshold region of the score (best split near 0.0063, **log-rank p=0.0032**, 701 versus 380 patients), and the reproduced course optimal-input procedure selected the highest DDR dose (**Cox HR 0.84 per SD, p=0.027**; optimal-cutpoint **log-rank p=0.0032**, 603 versus 478 patients). Because the threshold and dose are chosen to minimise the log-rank p-value, these results are not treated as confirmatory evidence; they indicate a possible threshold-like pattern rather than stronger evidence than the continuous Cox model.

The Q1 result is therefore interpreted as a modest prognostic association, weakening to borderline after stage adjustment, and not as a validated survival classifier. Importantly, this is a prognostic TCGA survival result, not direct evidence of doxorubicin response.

![Q1 TCGA survival by p53 S46 score](../figures/q1_p53_ode_tcga_km.png)

*Figure 2. Q1 TCGA-BRCA survival by p53 S46 AUC score (median split).*

![Q1 cutpoint scan](../figures/q1_p53_ode_tcga_cutpoint_scan.png)

*Figure 3. Q1 optimal cutpoint scan: log-rank p-value at each candidate threshold of the p53 S46 AUC score (log scale), with the selected cutpoint and the p=0.05 line marked. Candidate thresholds are restricted so each group keeps at least 10% of patients.*

![Q1 optimal cutpoint KM](../figures/q1_p53_ode_tcga_km_optimal_cutpoint.png)

*Figure 4. Q1 TCGA-BRCA survival by the optimal cutpoint split of the p53 S46 AUC score (701 versus 380 patients, log-rank p=0.0032).*

## Results Q2: p53 ODE Model and PRISM Doxorubicin Response

The same pre-specified p53 S46 AUC score showed no detectable association with PRISM doxorubicin sensitivity in the 26 breast cancer cell lines [3]: **Spearman r=0.04 (95% CI -0.37 to 0.44), p=0.838** and **Pearson r=0.01 (95% CI -0.38 to 0.40), p=0.961**. With only 26 cell lines the confidence intervals are wide, so this should be read as a lack of detectable association rather than as proof of no biological relationship; the analysis is underpowered for anything except a large effect.

This is consistent with the ODE S46 score capturing survival associated p53/DNA damage biology in TCGA-BRCA while not behaving as a direct doxorubicin sensitivity predictor in this small PRISM breast cancer subset.


![Q2 p53 ODE score versus PRISM doxorubicin sensitivity](../figures/q2_p53_ode_score_vs_prism_doxorubicin_sensitivity.png)

*Figure 5. Q2 p53 ODE score versus PRISM doxorubicin sensitivity.*

## Results Q3: PRISM Doxorubicin Signature Transferred to TCGA-BRCA Survival

Q3 fitted a simple elastic-net model [5] using the 16 shared p53/DNA-damage genes to predict PRISM doxorubicin sensitivity from breast cancer cell-line expression. The elastic-net penalty was fixed a priori (`alpha=0.1`, `l1_ratio=0.5`) rather than tuned, so there is no hyperparameter selection inside the cross-validation loop. The apparent in-sample fit looked moderate: **R²=0.44, RMSE=1.10 and Spearman r=0.76**. However, leave-one-out cross-validation was poor: **R²=-0.83, RMSE=1.99 and Spearman r=-0.22**. Because of the very small cohort, the leave-one-out result is used only as a conservative check against overinterpreting the in-sample fit.

When the PRISM-derived signature was applied to TCGA-BRCA, tumour expression was standardised within its own domain (rather than reusing the cell-line scaler) so the signature transfers as a relative within-domain score. The transferred score was not associated with overall survival: **HR 0.95 (95% CI 0.78-1.15), p=0.60**. The age-adjusted result was also null: **HR 0.95, p=0.61**, and neither a median split (**log-rank p=0.88**) nor the course's optimal-cutpoint split (**log-rank p=0.24**) separated survival.

The key interpretation is that apparent cell-line fit did not generalise, and the transferred score was not prognostic in TCGA-BRCA.

![Q3 leave-one-out cell-line prediction](../figures/q3_prism_signature_cellline_cv_predicted_vs_observed.png)

*Figure 6. Q3 leave-one-out PRISM prediction.*

Conceptual elastic-net signature score (coefficients learned in PRISM/DepMap cell lines):

```text
prism_signature_score = model_intercept
    + sum(gene_expression_z[g] * coefficient[g] for g in shared_genes)
```

## Results Q4: TCGA Survival Signature Transferred to PRISM Doxorubicin Response

Q4 fitted the reverse transfer, which is an extension beyond the course material: the taught transfer runs only from a cell-line drug-response signature to patient survival (the Q3 direction). A Cox model [4] based on the shared p53/DNA-damage genes produced a TCGA-derived survival-risk score. This score was strongly associated with survival within the training dataset: **HR 2.72, p=9.5e-08**; age-adjusted **HR 2.42, p=3.4e-06**; median-split **log-rank p=1.3e-06**; and the optimal-cutpoint split was similar (**log-rank p=1.9e-07**). This is expected and is not independent validation, because the score is fitted and tested on the same patients (a cross-validated or held-out estimate would be the appropriate next step).

When the TCGA-derived survival-risk score was applied to PRISM/DepMap cell lines (standardised within their own domain), it did not correlate with doxorubicin sensitivity: **Spearman r=0.07 (95% CI -0.35 to 0.46), p=0.74** and **Pearson r=0.04 (95% CI -0.35 to 0.42), p=0.84**. As in Q2, the small cell-line cohort means these wide intervals indicate a lack of detectable association rather than proof of none.

This means the patient survival-risk score was meaningful inside TCGA-BRCA, but it did not predict in vitro doxorubicin sensitivity.

![Q4 TCGA-derived risk score versus PRISM doxorubicin sensitivity](../figures/q4_tcga_score_vs_prism_doxorubicin_sensitivity.png)

*Figure 7. Q4 TCGA-derived risk score versus PRISM doxorubicin sensitivity.*

Conceptual TCGA Cox risk score (coefficients learned from TCGA-BRCA survival):

```text
tcga_survival_risk_score =
    sum(gene_expression_z[g] * cox_coefficient[g] for g in shared_genes)
```

## Summary of Findings Across Q1–Q4

| Question | Model | Training → test domain | Endpoint | Key result | Reading |
| --- | --- | --- | --- | --- | --- |
| Q1 | mechanistic p53 ODE | course template → TCGA-BRCA | overall survival | HR 0.85/SD (95% CI 0.73–0.99), p=0.038; stage-adjusted HR 0.85, p=0.058 | modest prognostic signal, partly stage-linked |
| Q2 | mechanistic p53 ODE | course template → PRISM | doxorubicin sensitivity | Spearman r=0.04 (95% CI −0.37–0.44), p=0.84 | no detectable association (underpowered) |
| Q3 | elastic-net signature | PRISM cell lines → TCGA-BRCA | overall survival | in-sample R²=0.44, LOOCV R²=−0.83; transfer HR 0.95 (95% CI 0.78–1.15), p=0.60 | apparent fit did not generalise or transfer |
| Q4 | Cox gene signature | TCGA-BRCA → PRISM | doxorubicin sensitivity | within-TCGA HR 2.72, p<1e-7; transfer Spearman r=0.07 (95% CI −0.35–0.46), p=0.74 | strong within-domain, no transfer |

This is the mechanistic-versus-regression comparison that the original Q5 asked for. Across all four questions the strongest evidence was within-domain, and no signal transferred cleanly between patient survival and in vitro doxorubicin response. The mechanistic ODE score (Q1–Q2) and the expression-based regression signatures (Q3–Q4) behaved similarly: useful or modest within their training endpoint, but not transferable across endpoints.

## Discussion

Across Q1-Q4, the clearest pattern was that endpoint context mattered. TCGA-BRCA survival [1] and PRISM doxorubicin sensitivity [3] are biologically related but not equivalent. TCGA survival reflects tumour biology, treatment heterogeneity, patient age, stage, immune context, comorbidity and follow-up. PRISM measures in vitro relative abundance after drug treatment in cell lines, without tumour microenvironment or patient-level clinical complexity.

The p53 ODE score showed a modest survival association in TCGA-BRCA but no PRISM doxorubicin sensitivity association. This suggests that p53/DNA-damage response expression may carry some prognostic information without directly predicting doxorubicin response. Similarly, the Q3 and Q4 transfer analyses were negative: a PRISM-derived drug-response signature did not transfer to patient survival, and a TCGA-derived survival score did not transfer to cell-line doxorubicin sensitivity.

The Q3 in-sample result is a useful warning. The elastic-net model looked moderately predictive when assessed on the same 26 cell lines used for fitting, but leave-one-out cross-validation was poor. This illustrates how in-sample model fit can be misleading when sample size is small.

The weak cross-domain results are biologically and statistically plausible rather than simply negative. TCGA-BRCA overall survival is a broad prognostic endpoint, not a direct measure of doxorubicin benefit, whereas PRISM measures in vitro cell-line viability after doxorubicin exposure without patient-level treatment context, tumour microenvironment or clinical follow-up. The p53 S46 ODE score may therefore capture a general DNA-damage or apoptosis-associated tumour state that is modestly prognostic in TCGA-BRCA, without being a direct predictor of doxorubicin sensitivity; consistent with this, the survival association weakened to borderline once tumour stage was included. In addition, breast cancer is molecularly heterogeneous, and pooling ER-positive, HER2-positive and basal-like cases may average away a signal that matters more in one subtype. A 16-gene p53/DNA-damage panel may also miss important determinants of anthracycline response, including proliferation, drug transport, broader DNA repair and apoptosis context, and the small PRISM breast cancer cohort further limits power to detect anything other than large drug-response associations. The main conclusion is therefore not that p53/DNA-damage biology is unimportant, but that this simplified baseline-expression and ODE-score representation does not transfer cleanly between patient survival and cell-line doxorubicin response.

Negative transfer results are still informative. They show that a signature can be meaningful for one endpoint and unhelpful for another, which is a central issue in personalised medicine modelling.

## Limitations

The PRISM/DepMap analysis had only 26 breast cancer cell lines after intersecting lineage, expression, treatment metadata and non-missing doxorubicin LFC. This small sample size limits model stability and makes cross-validation estimates noisy.

TCGA overall survival is not direct doxorubicin response. TCGA-BRCA also includes treatment heterogeneity, and the available survival endpoint does not isolate anthracycline benefit. Age and stage were available, but the modelling remained deliberately simple for a transparent assignment analysis.

The gene set was restricted to 16 selected p53/DNA-damage genes. This keeps the assignment explainable but may miss other determinants of doxorubicin response, such as drug transport, proliferation, DNA repair beyond the selected genes and apoptosis context.

The p53 ODE model was adapted from the given template and was not fully re-calibrated for breast cancer. TCGA and DepMap/CCLE are static baseline expression snapshots, not p53 dynamic time-course measurements. There was no external validation dataset.

Some statistical caveats also apply. The continuous Cox model satisfied the proportional-hazards assumption (Schoenfeld test p=0.21), but the Kaplan-Meier curves become unstable in the late follow-up tail where few patients remain at risk, so late separation should not be over-interpreted. The optimal cutpoint and optimal-dose analyses are data-driven and optimistic by construction and are reported as exploratory only. The PRISM correlation analyses are underpowered at n=26, with wide confidence intervals, so the cell-line results indicate a lack of detectable association rather than evidence of no effect. The TCGA survival-risk score in Q4 is reported only as a within-sample association and was not internally cross-validated.

## Conclusion

The analyses suggest that p53/DNA-damage response features can show modest prognostic signal in TCGA-BRCA, but neither mechanistic ODE features nor expression-based signatures transferred cleanly between patient survival and PRISM doxorubicin sensitivity. This highlights an important personalised medicine lesson: in vitro drug response and patient survival are related but non-equivalent endpoints.

## References

1. The Cancer Genome Atlas Network. Comprehensive molecular portraits of human breast tumours. *Nature*. 2012.
2. Ghandi M, Huang FW, Jané-Valbuena J, et al. Cancer Cell Line Encyclopedia study. *Nature*. 2019.
3. Corsello SM, Nagari RT, Spangler RD, et al. PRISM drug viability profiling study. *Nature Cancer*. 2020.
4. Cox DR. Regression models and life-tables. *Journal of the Royal Statistical Society: Series B*. 1972.
5. Zou H and Hastie T. Elastic net regression method. *Journal of the Royal Statistical Society: Series B*. 2005.